## Data Pipeline Overview

This script uses the PRAW library to connect to Reddit and fetch recent posts from the subreddits "anxiety", "depression", and "adhd". It retrieves posts in batches, filters out duplicates, and only includes posts from the last 10 years. For each post, it extracts details like the author's name, creation time, score, title, and selftext content. The script handles potential API errors with retry logic and pauses to avoid rate limits. Once all data is collected, it saves the results in `reddit_data.csv` for further acquistion.

This script processes Reddit post data by cleaning, tokenizing, and lemmatizing the text content from `reddit_data.csv`. It first ensures necessary NLP resources are available using `nltk` and `spaCy`, then tokenizes the selftext (main post content), removes stopwords and punctuation, and quotes each word. It also lemmatizes the cleaned tokens to reduce words to their base forms. If a previously processed file (`merged_output_tokenized.csv`) exists, it merges the new data while avoiding duplicates. Finally, it saves the cleaned and merged dataset back to a CSV file.



In [ ]:
!pip install praw
!pip install schedule

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 6.6 MB/s eta 0:00:00


In [ ]:
import praw
import pandas as pd
import time
import os
from datetime import datetime, timedelta

reddit = praw.Reddit(
    client_id="TIyXs6Cidt4b_Iut84v0jw",
    client_secret="r7JPzh046bVXuBqKlQbmhVAniwev7Q",
    user_agent="macOS:sample testing:v1.0.0 (by u/ostwalaman)"
)

subreddits = ["anxiety", "depression", "adhd"]

posts_per_batch = 10000

def fetch_reddit_data():
    print("Starting Reddit data fetch...")
    data = []
    unique_post_ids = set()
    one_year_ago = datetime.utcnow() - timedelta(days=3650)
    one_year_ago_timestamp = int(one_year_ago.timestamp())

    for subreddit_name in subreddits:
        subreddit = reddit.subreddit(subreddit_name)
        last_post_time = None

        while True:
            try:
                posts = subreddit.new(limit=posts_per_batch, params={"before": last_post_time} if last_post_time else {})
                posts_fetched = 0

                for post in posts:
                    if post.created_utc < one_year_ago_timestamp:
                        break
                    if post.id in unique_post_ids:
                        continue
                    unique_post_ids.add(post.id)

                    created_datetime = datetime.utcfromtimestamp(post.created_utc).strftime('%Y-%m-%d %H:%M:%S')

                    data.append({
                        "Author": post.author.name if post.author else None,
                        "Created_DateTime": created_datetime,
                        "Score": post.score,
                        "Selftext": post.selftext,
                        "Subreddit": post.subreddit.display_name,
                        "Title": post.title
                    })
                    last_post_time = post.created_utc
                    posts_fetched += 1

                if posts_fetched == 0:
                    break
            except Exception as e:
                print(f"Error fetching posts from r/{subreddit_name}: {e}")
                time.sleep(5)

            time.sleep(1)

    df = pd.DataFrame(data)

    df.to_csv("reddit_data.csv", index=False)
print("Data saved as reddit_data.csv")

Data saved as reddit_data.csv


In [ ]:
fetch_reddit_data()

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Starting Reddit data fetch...


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/l

In [ ]:
pd.read_csv("reddit_data.csv")

,Author,Created_DateTime,Score,Selftext,Subreddit,Title
0,Goose_gal420,2025-03-23 00:19:42,1,Hi so I was at the ER. (For a headache). They ...,Anxiety,Anyone ever had this happen to them?
1,Old_Device_3,2025-03-23 00:15:32,1,\n\n(if you've seen this alot before and are g...,Anxiety,Struggling
2,Omar_Qaddura,2025-03-23 00:10:07,1,I (18m) have been experiencing a ton of severe...,Anxiety,How to stop frequent panic attacks?
3,throwaway999999870,2025-03-23 00:07:39,2,I absolutely loathe this woman for a lot of re...,Anxiety,My Boss Has Become Extremely Passive Aggressiv...
4,notasheeplol_,2025-03-23 00:05:59,1,"Hi all, I thought I would share my experience ...",Anxiety,My experience with severe anxiety.
...,...,...,...,...,...,...
2955,L_Merce_001,2025-03-16 11:32:39,2,Disclaimer: I am aware that smoking is bad for...,ADHD,I want to smoke constantly
2956,kinkade,2025-03-16 11:18:23,2,I was diagnosed with ADHD as a 45-year-old adu...,ADHD,Adult Intuniv Tiredness
2957,Concentrate_Amazing,2025-03-16 11:02:59,1,"First of all, I have no real trouble living a ...",ADHD,Should I seek an opinion
2958,tiredbuthavegoals,2025-03-16 10:39:09,2,So we all know that we have a very “bad” memor...,ADHD,How vividly do you recall things / events?


In [ ]:
import os
import pandas as pd
import nltk
import spacy
import time
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import csv

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

nlp = spacy.load("en_core_web_sm")

scraped_file = "/content/reddit_data.csv"  # New fetched data
processed_output_file = "/content/merged_output_tokenized.csv"  # Final processed output

def clean_and_tokenize(text):
    """Tokenize and clean text by removing stopwords and punctuation, and quote each word."""
    if pd.isna(text):
        return ""
    text = text.lower()
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [f'"{word}"' for word in tokens if word.isalpha() and word not in stop_words]  # Quote each word
    return " ".join(tokens)

def lemmatize_text(text):
    """Perform lemmatization on the text using spaCy and quote each lemmatized word."""
    if isinstance(text, str):
        doc = nlp(text)
        return " ".join([f'"{token.lemma_}"' for token in doc])  # Quote each lemmatized word
    return text

def clean_process_and_merge_data():

    if not os.path.exists(scraped_file):
        print("No new scraped data found.")
        return

    df = pd.read_csv(scraped_file)

    df = df.dropna(subset=["Selftext"])

    df["Author"].fillna("Anonymous", inplace=True)

    print("Processing new data...")
    df["New_Token_Selftext"] = df["Selftext"].apply(clean_and_tokenize)
    df["Lemmatized_Text"] = df["New_Token_Selftext"].apply(lemmatize_text)

    if os.path.exists(processed_output_file):
        existing_df = pd.read_csv(processed_output_file)
        print("Merging with existing processed dataset...")

        final_df = pd.concat([existing_df, df], ignore_index=True)
        final_df.drop_duplicates(subset=["Title", "Selftext"], keep="last", inplace=True)
    else:
        print("No existing processed file found. Creating a new one...")
        final_df = df

    final_df.to_csv(processed_output_file, index=False, quoting=csv.QUOTE_MINIMAL)  # Ensures words remain quoted

    print(f"Processed and merged data saved successfully as {processed_output_file}!")

clean_process_and_merge_data()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
<ipython-input-15-593387a2f476>:46: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Author"].fillna("Anonymous", inplace=True)


Processing new data...
Merging with existing processed dataset...
Processed and merged data saved successfully as /content/merged_output_tokenized.csv!


In [ ]:
import pandas as pd
import os

merged_tokenized_file = "/content/merged_output_tokenized.csv"

def check_merged_tokenized_data():
    if os.path.exists(merged_tokenized_file):
        df = pd.read_csv(merged_tokenized_file)

        row_count = df.shape[0]

        column_count = df.shape[1]

        print(f"Merged Tokenized Data Count (Rows): {row_count}")
        print(f"Total Column Count: {column_count}")

        print("\nColumn Names and Non-Null Counts:")
        print(df.count())

    else:
        print("Merged output tokenized file not found.")

check_merged_tokenized_data()

Merged Tokenized Data Count (Rows): 22273
Total Column Count: 8

Column Names and Non-Null Counts:
Author                22273
Created_DateTime      22273
Score                 22273
Selftext              22273
Subreddit             22273
Title                 22273
New_Token_Selftext    22252
Lemmatized_Text       22252
dtype: int64


In [ ]:
import pandas as pd

file_path = "/content/merged_output_tokenized.csv"
df = pd.read_csv(file_path)

df.head(10)

,Author,Created_DateTime,Score,Selftext,Subreddit,Title,New_Token_Selftext,Lemmatized_Text
0,mamboitalianoo,2025-01-31 21:32:39,1,A couple years ago I had a bout of panic attac...,Anxiety,Panic attack or vertigo?,"""couple"" ""years"" ""ago"" ""bout"" ""panic"" ""attacks...","""couple"" ""year"" ""ago"" ""bout"" ""panic"" ""attack"" ..."
1,Midori_sho,2025-01-31 21:31:03,1,School has been kicking my ass lately. My midt...,Anxiety,I think I’m actually stupid as hell,"""school"" ""kicking"" ""ass"" ""lately"" ""midterms"" ""...","""school"" ""kicking"" ""ass"" ""lately"" ""midterm"" ""k..."
2,No_Entrepreneur_4395,2025-01-31 21:29:09,1,"I've never been diagnosed by a doctor, but I'm...",Anxiety,Is being medicated better?,"""never"" ""diagnosed"" ""doctor"" ""pretty"" ""sure"" ""...","""never"" ""diagnose"" ""doctor"" ""pretty"" ""sure"" ""a..."
3,Successful-Bat-5287,2025-01-31 21:22:31,1,Long time anxiety sufferer here. I’ve done the...,Anxiety,My body is anxious but I’m not?,"""long"" ""time"" ""anxiety"" ""sufferer"" ""done"" ""the...","""long"" ""time"" ""anxiety"" ""sufferer"" ""do"" ""thera..."
4,ratherdream,2025-01-31 21:07:44,1,I'm so afraid of losing the ones I love and my...,Anxiety,Afraid to loose people,"""afraid"" ""losing"" ""ones"" ""love"" ""insecurities""...","""afraid"" ""lose"" ""one"" ""love"" ""insecurity"" ""mak..."
5,gayLiam116111,2025-01-31 21:01:59,2,"Hi, im 15f and since november of last year i h...",Anxiety,How do i stop thinking that im dying constantly?,"""hi"" ""im"" ""since"" ""november"" ""last"" ""year"" ""ma...","""hi"" ""I"" ""m"" ""since"" ""november"" ""last"" ""year"" ..."
6,breenotsoswag,2025-01-31 20:56:06,2,i did it! there are mixed reviews about this p...,Anxiety,i made my psychiatry appointment,"""mixed"" ""reviews"" ""psychiatrist"" ""butttt"" ""gla...","""mixed"" ""review"" ""psychiatrist"" ""butttt"" ""glad..."
7,okokayOKokayk,2025-01-31 20:51:02,4,I got an assessment recently and was diagnosed...,Anxiety,Can Low Motivation be Caused by Anxiety?,"""got"" ""assessment"" ""recently"" ""diagnosed"" ""gad...","""get"" ""assessment"" ""recently"" ""diagnose"" ""gad""..."
8,succeedathumanity,2025-01-31 20:28:32,1,"I am an assistant manager of a restaurant, an...",Anxiety,My General manager cries at work weekly.,"""assistant"" ""manager"" ""restaurant"" ""general"" ""...","""assistant"" ""manager"" ""restaurant"" ""general"" ""..."
9,Able-Top1151,2025-01-31 20:25:39,1,Crashing out saw a Tabasco bottle and fucking ...,Anxiety,Adrenaline anger rushed read what I did,"""crashing"" ""saw"" ""tabasco"" ""bottle"" ""fucking"" ...","""crash"" ""see"" ""tabasco"" ""bottle"" ""fucking"" ""ch..."
